# NB08 - FeParam metadata extraction

## Purpose

This notebook explores whether Mindray TE7 native `VirtualMachine.txt` and `VirtualMachine.bin`
contain useful acquisition metadata in FeParam blocks.

The goal is to extract and organize metadata-like information that may describe acquisition settings,
for example power, focus, dynamic range, beam parameters, mode parameters, and related machine settings.

## Scope

### In scope

- Discover `VirtualMachine.txt` and `VirtualMachine.bin` for each native recording.
- Parse FeParam offsets and declared sizes when available.
- Extract PW and BC FeParam binary blobs.
- Inventory readable strings and parameter names inside the blobs.
- Compare FeParam structure across recordings.
- Export recording-level and parameter-level metadata tables.

### Out of scope

- No clinical interpretation.
- No velocity envelope extraction.
- No PSV, EDV, RI, PI, VTI, SV, CO, or BP.
- No stiffness or compliance estimation.
- No native beat timing.
- No claim that a parameter has known meaning unless validated.

## Expected outputs

```text
reports/nb08_feparam/nb08_feparam_blob_inventory.csv
reports/nb08_feparam/nb08_feparam_string_inventory.csv
reports/nb08_feparam/nb08_feparam_parameter_name_summary.csv
docs/nb08_feparam/nb08_feparam_summary.md

## Imports & Setup

In [3]:
from pathlib import Path
import re
import math
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

In [4]:
def resolve_project_root(candidate_roots=None):
    """
    Resolve the DopplerLab project root.

    Parameters
    ----------
    candidate_roots : list[pathlib.Path] or None
        Candidate project roots.

    Returns
    -------
    pathlib.Path
        Resolved project root.

    Raises
    ------
    FileNotFoundError
        If no candidate root exists.
    """
    if candidate_roots is None:
        candidate_roots = [
            Path(r"E:\DopplerLab"),
            Path(r"D:\code\DopplerLab"),
            Path.cwd(),
            Path.cwd().parent,
        ]

    checked = []

    for root in candidate_roots:
        root = Path(root)
        checked.append(str(root))

        if not root.exists():
            continue

        expected_any = [
            root / "ultrasound_recordings",
            root / "reports",
            root / "scope",
        ]

        if any(path.exists() for path in expected_any):
            return root

    raise FileNotFoundError(
        "Could not resolve DopplerLab project root. Checked:\n"
        + "\n".join(checked)
    )


def build_nb08_paths(project_root):
    """
    Build paths for NB08 FeParam metadata extraction.

    Parameters
    ----------
    project_root : pathlib.Path
        DopplerLab project root.

    Returns
    -------
    dict
        Named paths.
    """
    project_root = Path(project_root)

    paths = {
        "PROJECT_ROOT": project_root,
        "NATIVE_BATCH_DIR": project_root / "ultrasound_recordings" / "batch_2026_06_13_native",

        "NB08_REPORTS_DIR": project_root / "reports" / "nb08_feparam",
        "NB08_DOCS_DIR": project_root / "docs" / "nb08_feparam",
        "NB08_FIGURES_DIR": project_root / "figures" / "nb08_feparam",

        # Helpful previous outputs, not mandatory.
        "NB06_V2_COMPACT_CONTROL_TABLE": project_root / "reports" / "nb06_v2" / "nb06_v2_compact_control_table.csv",
        "NB06_V2_DIAGNOSTIC_TABLE": project_root / "reports" / "nb06_v2" / "nb06_v2_recording_diagnostic.csv",
        "SCOPE04_FEPARAM_FORENSICS_CSV": project_root / "scope" / "04_physio_feparam_header_forensics" / "reports" / "feparam_forensics.csv",
        "SCOPE04_AUXILIARY_DOC": project_root / "scope" / "04_physio_feparam_header_forensics" / "docs" / "NATIVE_AUXILIARY_FORENSICS.md",
    }

    return paths


def ensure_nb08_output_dirs(paths):
    """
    Create NB08 output directories.

    Parameters
    ----------
    paths : dict
        Path dictionary.
    """
    for key in [
        "NB08_REPORTS_DIR",
        "NB08_DOCS_DIR",
        "NB08_FIGURES_DIR",
    ]:
        paths[key].mkdir(parents=True, exist_ok=True)


def build_path_status_table(paths):
    """
    Build a path status table.

    Parameters
    ----------
    paths : dict
        Named paths.

    Returns
    -------
    pandas.DataFrame
        Path status table.
    """
    required = {
        "PROJECT_ROOT",
        "NATIVE_BATCH_DIR",
    }

    optional = {
        "NB06_V2_COMPACT_CONTROL_TABLE",
        "NB06_V2_DIAGNOSTIC_TABLE",
        "SCOPE04_FEPARAM_FORENSICS_CSV",
        "SCOPE04_AUXILIARY_DOC",
    }

    outputs = {
        "NB08_REPORTS_DIR",
        "NB08_DOCS_DIR",
        "NB08_FIGURES_DIR",
    }

    rows = []

    for key, path in paths.items():
        path = Path(path)

        if key in required:
            role = "required"
        elif key in optional:
            role = "optional_previous_output"
        elif key in outputs:
            role = "output"
        else:
            role = "support"

        rows.append(
            {
                "key": key,
                "role": role,
                "exists": path.exists(),
                "is_dir": path.is_dir() if path.exists() else False,
                "path": str(path),
            }
        )

    return pd.DataFrame(rows).sort_values(["role", "key"]).reset_index(drop=True)


def print_nb08_setup_summary(project_root, paths, status_df):
    """
    Print setup summary.

    Parameters
    ----------
    project_root : pathlib.Path
        Project root.

    paths : dict
        Named paths.

    status_df : pandas.DataFrame
        Path status table.
    """
    print("NB08 FeParam setup")
    print("=" * 80)
    print(f"PROJECT_ROOT: {project_root}")
    print()
    
    missing_required = status_df[(status_df["role"] == "required") & (~status_df["exists"])]

    print(f"Missing required paths: {len(missing_required)}")
    print()

    if len(missing_required) > 0:
        display(missing_required)

    print("Output directories:")
    for key in ["NB08_REPORTS_DIR", "NB08_DOCS_DIR", "NB08_FIGURES_DIR"]:
        print(f"  - {key}: {paths[key]}")


In [5]:
PROJECT_ROOT = resolve_project_root()
PATHS = build_nb08_paths(PROJECT_ROOT)
ensure_nb08_output_dirs(PATHS)

path_status_df = build_path_status_table(PATHS)

print_nb08_setup_summary(
    project_root=PROJECT_ROOT,
    paths=PATHS,
    status_df=path_status_df,
)

path_status_df

NB08 FeParam setup
PROJECT_ROOT: E:\DopplerLab

Missing required paths: 0

Output directories:
  - NB08_REPORTS_DIR: E:\DopplerLab\reports\nb08_feparam
  - NB08_DOCS_DIR: E:\DopplerLab\docs\nb08_feparam
  - NB08_FIGURES_DIR: E:\DopplerLab\figures\nb08_feparam


,key,role,exists,is_dir,path
0,NB06_V2_COMPACT_CONTROL_TABLE,optional_previous_output,True,False,E:\DopplerLab\reports\nb06_v2\nb06_v2_compact_...
1,NB06_V2_DIAGNOSTIC_TABLE,optional_previous_output,True,False,E:\DopplerLab\reports\nb06_v2\nb06_v2_recordin...
2,SCOPE04_AUXILIARY_DOC,optional_previous_output,True,False,E:\DopplerLab\scope\04_physio_feparam_header_f...
3,SCOPE04_FEPARAM_FORENSICS_CSV,optional_previous_output,True,False,E:\DopplerLab\scope\04_physio_feparam_header_f...
4,NB08_DOCS_DIR,output,True,True,E:\DopplerLab\docs\nb08_feparam
5,NB08_FIGURES_DIR,output,True,True,E:\DopplerLab\figures\nb08_feparam
6,NB08_REPORTS_DIR,output,True,True,E:\DopplerLab\reports\nb08_feparam
7,NATIVE_BATCH_DIR,required,True,True,E:\DopplerLab\ultrasound_recordings\batch_2026...
8,PROJECT_ROOT,required,True,True,E:\DopplerLab


## FeParam file discovery and blob inventory

This section discovers `VirtualMachine.txt` and `VirtualMachine.bin` for each native recording.

It then tries to identify FeParam blobs using two layers:

1. Parse offsets and sizes from `VirtualMachine.txt` if available.
2. Use known validated fallback offsets from Scope 04 only when text parsing is unavailable.

Important:

Fallback offsets are allowed here only as an exploratory starting point.
They are not treated as a general Mindray format specification.

In [6]:
# Scope 04 observed these offsets/sizes in this batch.
# Use only as fallback if VirtualMachine.txt parsing does not recover them.
FALLBACK_FEPARAM_LAYOUT = {
    "PW": {"offset": 0, "size": 4514},
    "BC": {"offset": 4530, "size": 30935},
}


In [9]:
def safe_file_size(path):
    """
    Return file size in bytes or None.

    Parameters
    ----------
    path : pathlib.Path or str
        File path.

    Returns
    -------
    int or None
        File size.
    """
    path = Path(path)

    if not path.exists() or not path.is_file():
        return None

    return int(path.stat().st_size)


def compute_shannon_entropy(byte_data):
    """
    Compute Shannon entropy of byte data.

    Parameters
    ----------
    byte_data : bytes
        Binary data.

    Returns
    -------
    float or None
        Entropy in bits per byte.
    """
    if byte_data is None or len(byte_data) == 0:
        return None

    counts = np.bincount(np.frombuffer(byte_data, dtype=np.uint8), minlength=256)
    probabilities = counts[counts > 0] / len(byte_data)
    entropy = -np.sum(probabilities * np.log2(probabilities))

    return float(entropy)


def discover_native_recording_dirs(native_batch_dir):
    """
    Discover native recording directories.

    Parameters
    ----------
    native_batch_dir : pathlib.Path or str
        Native batch directory.

    Returns
    -------
    list[pathlib.Path]
        Sorted recording directories.
    """
    native_batch_dir = Path(native_batch_dir)

    if not native_batch_dir.exists():
        raise FileNotFoundError(f"Native batch directory not found: {native_batch_dir}")

    recording_dirs = [path for path in sorted(native_batch_dir.iterdir()) if path.is_dir()]

    return recording_dirs


def extract_ascii_strings(byte_data, min_length=4):
    """
    Extract printable ASCII strings from binary data.

    Parameters
    ----------
    byte_data : bytes
        Binary data.

    min_length : int
        Minimum string length.

    Returns
    -------
    list[str]
        Extracted strings.
    """
    if byte_data is None:
        return []

    pattern = rb"[ -~]{" + str(min_length).encode("ascii") + rb",}"
    matches = re.findall(pattern, byte_data)
    strings = [match.decode("ascii", errors="replace") for match in matches]

    return strings


def parse_feparam_param_sizes_from_virtual_machine_txt(vm_txt_path):
    """
    Parse FeParam parameter block sizes from VirtualMachine.txt.

    This parser avoids broad numeric guessing.

    Expected observed structure in this batch:
    - VirtualMachine.txt contains FeParam-like param blocks.
    - param_0 size corresponds to PW FeParam size.
    - param_1 size corresponds to BC FeParam size.
    - VirtualMachine.bin stores:
        PW blob at offset 0
        separator/header gap of 16 bytes
        BC blob at offset PW_size + 16

    Parameters
    ----------
    vm_txt_path : pathlib.Path or str
        Path to VirtualMachine.txt.

    Returns
    -------
    dict
        Parsed sizes and source information.
    """
    vm_txt_path = Path(vm_txt_path)

    result = {
        "parse_ok": False,
        "error": "",
        "param_sizes": {},
        "source": "virtual_machine_txt_param_blocks",
    }

    if not vm_txt_path.exists():
        result["error"] = "file_not_found"
        return result

    try:
        text = vm_txt_path.read_text(encoding="utf-8", errors="replace",)
    except Exception as exc:
        result["error"] = f"read_error: {exc}"
        return result

    # Parse blocks like:
    # DATA_TREE_BEGIN=param_0
    #     ...
    #     SIZE=4514
    # DATA_TREE_END=param_0
    #
    # Keep this specific; do not scan broad unrelated windows.
    block_pattern = re.compile(r"DATA_TREE_BEGIN=(param_\d+)(.*?)DATA_TREE_END=\1", flags=re.DOTALL,)

    for match in block_pattern.finditer(text):
        param_name = match.group(1)
        block = match.group(2)

        size_match = re.search(r"\bSIZE\s*=\s*(\d+)", block, flags=re.IGNORECASE)

        if size_match is None:
            continue

        result["param_sizes"][param_name] = {"size": int(size_match.group(1)), "source_text": block[:300].replace("\n", " ").replace("\t", " ")}

    if len(result["param_sizes"]) >= 2:
        result["parse_ok"] = True
    else:
        result["error"] = f"expected_at_least_2_param_sizes_found_{len(result['param_sizes'])}"

    return result


def get_feparam_layout(vm_txt_path, vm_bin_path=None, allow_fallback=True):
    """
    Get FeParam layout from VirtualMachine.txt and VirtualMachine.bin size.

    Correct NB08 rule:

    - PW FeParam:
        offset = 0
        size = SIZE from param_0

    - BC FeParam:
        offset = PW_size + 16
        size = SIZE from param_1 if available and consistent,
               otherwise vm_bin_size - BC_offset.

    Parameters
    ----------
    vm_txt_path : pathlib.Path or str
        Path to VirtualMachine.txt.

    vm_bin_path : pathlib.Path or str or None
        Path to VirtualMachine.bin. Used for consistency checks.

    allow_fallback : bool
        Whether to use conservative fallback if text parsing fails.

    Returns
    -------
    dict
        Layout dictionary for PW and BC.
    """
    vm_txt_path = Path(vm_txt_path)
    vm_bin_path = Path(vm_bin_path) if vm_bin_path is not None else None
    vm_bin_size = safe_file_size(vm_bin_path) if vm_bin_path is not None else None
    parsed = parse_feparam_param_sizes_from_virtual_machine_txt(vm_txt_path)

    layout = {}

    if parsed["parse_ok"]:
        param_sizes = parsed["param_sizes"]
        param_names_sorted = sorted(param_sizes.keys(), key=lambda name: int(name.split("_")[1]),)
        pw_param = param_names_sorted[0]
        bc_param = param_names_sorted[1]
        pw_size = int(param_sizes[pw_param]["size"])
        pw_offset = 0
        bc_offset = pw_size + 16
        bc_size_from_txt = int(param_sizes[bc_param]["size"])
        bc_size_from_file = None

        if vm_bin_size is not None:
            bc_size_from_file = vm_bin_size - bc_offset

        # Prefer file-consistent BC size.
        # If text size matches it, great. If not, keep both in source text note.
        if bc_size_from_file is not None and bc_size_from_file > 0:
            bc_size = bc_size_from_file
            bc_size_source = "vm_bin_size_minus_bc_offset"
        else:
            bc_size = bc_size_from_txt
            bc_size_source = "virtual_machine_txt_param_1_size"

        layout["PW"] = {
            "offset": pw_offset,
            "size": pw_size,
            "source": "parsed_param_0_size_offset_0",
            "source_text": param_sizes[pw_param]["source_text"],
            "txt_size": pw_size,
            "file_consistent_size": pw_size,
        }

        layout["BC"] = {
            "offset": bc_offset,
            "size": bc_size,
            "source": f"parsed_param_1_size_with_{bc_size_source}",
            "source_text": param_sizes[bc_param]["source_text"],
            "txt_size": bc_size_from_txt,
            "file_consistent_size": bc_size_from_file,
        }

        return layout

    if allow_fallback:
        # Conservative fallback:
        # Use Scope04 observed PW size only when text parsing fails.
        # BC size is derived from actual VirtualMachine.bin size if possible.
        pw_size = FALLBACK_FEPARAM_LAYOUT["PW"]["size"]
        pw_offset = 0
        bc_offset = pw_size + 16

        if vm_bin_size is not None and vm_bin_size > bc_offset:
            bc_size = vm_bin_size - bc_offset
            bc_source = "fallback_pw_size_bc_size_from_vm_bin_size"
        else:
            bc_size = FALLBACK_FEPARAM_LAYOUT["BC"]["size"]
            bc_source = "scope04_static_fallback"

        layout["PW"] = {
            "offset": pw_offset,
            "size": pw_size,
            "source": "scope04_fallback_pw_size_offset_0",
            "source_text": parsed.get("error", ""),
            "txt_size": None,
            "file_consistent_size": pw_size,
        }

        layout["BC"] = {
            "offset": bc_offset,
            "size": bc_size,
            "source": bc_source,
            "source_text": parsed.get("error", ""),
            "txt_size": None,
            "file_consistent_size": bc_size,
        }

    return layout


def extract_feparam_blob(vm_bin_path, offset, size):
    """
    Extract a FeParam blob from VirtualMachine.bin.

    Parameters
    ----------
    vm_bin_path : pathlib.Path or str
        Path to VirtualMachine.bin.

    offset : int
        Blob offset.

    size : int
        Blob size in bytes.

    Returns
    -------
    bytes
        Extracted blob.
    """
    vm_bin_path = Path(vm_bin_path)

    data = vm_bin_path.read_bytes()

    end = offset + size

    if offset < 0 or size <= 0 or end > len(data):
        raise ValueError(
            f"Invalid blob range for {vm_bin_path}: "
            f"offset={offset}, size={size}, file_size={len(data)}"
        )

    return data[offset:end]


def build_feparam_blob_inventory(native_batch_dir):
    """
    Build FeParam blob inventory for all native recordings.

    Parameters
    ----------
    native_batch_dir : pathlib.Path or str
        Native batch directory.

    Returns
    -------
    pandas.DataFrame
        One row per recording/mode blob.
    """
    recording_dirs = discover_native_recording_dirs(native_batch_dir)

    rows = []

    for recording_dir in recording_dirs:
        recording_id = recording_dir.name
        native_dir = recording_dir / "native"

        vm_txt_path = native_dir / "VirtualMachine.txt"
        vm_bin_path = native_dir / "VirtualMachine.bin"

        vm_txt_exists = vm_txt_path.exists()
        vm_bin_exists = vm_bin_path.exists()
        vm_bin_size = safe_file_size(vm_bin_path)

        layout = get_feparam_layout(
            vm_txt_path=vm_txt_path,
            vm_bin_path=vm_bin_path,
            allow_fallback=True,
        )

        for mode in ["PW", "BC"]:
            mode_layout = layout.get(mode, {})

            offset = mode_layout.get("offset")
            size = mode_layout.get("size")
            source = mode_layout.get("source", "")
            source_text = mode_layout.get("source_text", "")

            blob_extract_ok = False
            blob_error = ""
            entropy = None
            ascii_string_count = None
            first_strings = []

            if vm_bin_exists and offset is not None and size is not None:
                try:
                    blob = extract_feparam_blob(
                        vm_bin_path=vm_bin_path,
                        offset=offset,
                        size=size,
                    )

                    blob_extract_ok = True
                    entropy = compute_shannon_entropy(blob)

                    strings = extract_ascii_strings(
                        blob,
                        min_length=4,
                    )

                    ascii_string_count = len(strings)
                    first_strings = strings[:20]

                except Exception as exc:
                    blob_error = str(exc)

            else:
                blob_error = "missing_vm_bin_or_layout"

            rows.append(
                {
                    "recording_id": recording_id,
                    "mode": mode,
                    "vm_txt_exists": vm_txt_exists,
                    "vm_bin_exists": vm_bin_exists,
                    "vm_bin_size_bytes": vm_bin_size,
                    "feparam_offset": offset,
                    "feparam_size_bytes": size,
                    "layout_source": source,
                    "layout_txt_size": mode_layout.get("txt_size"),
                    "layout_file_consistent_size": mode_layout.get("file_consistent_size"),
                    "layout_source_text": source_text,
                    "blob_extract_ok": blob_extract_ok,
                    "blob_error": blob_error,
                    "blob_entropy": round(float(entropy), 4) if entropy is not None else None,
                    "ascii_string_count": ascii_string_count,
                    "first_strings_preview": " | ".join(first_strings[:8]),
                    "vm_txt_path": str(vm_txt_path),
                    "vm_bin_path": str(vm_bin_path),
                }
            )

    inventory_df = pd.DataFrame(rows)

    return inventory_df.sort_values(["recording_id", "mode"]).reset_index(drop=True)


def summarize_feparam_blob_inventory(blob_inventory_df):
    """
    Print summary of FeParam blob inventory.

    Parameters
    ----------
    blob_inventory_df : pandas.DataFrame
        Blob inventory table.
    """
    print("FeParam blob inventory")
    print("=" * 80)
    print(f"Rows: {len(blob_inventory_df)}")
    print()
    print("blob_extract_ok counts:")
    print(blob_inventory_df["blob_extract_ok"].value_counts(dropna=False))
    print()
    print("layout_source counts:")
    print(blob_inventory_df["layout_source"].value_counts(dropna=False))
    print()

    summary_cols = [
        "recording_id",
        "mode",
        "vm_bin_size_bytes",
        "feparam_offset",
        "feparam_size_bytes",
        "layout_source",
        "blob_extract_ok",
        "blob_entropy",
        "ascii_string_count",
        "first_strings_preview",
    ]

    display(blob_inventory_df[summary_cols])

    print()
    print("Unique offsets/sizes:")
    for mode in ["PW", "BC"]:
        mode_df = blob_inventory_df[blob_inventory_df["mode"] == mode]
        offsets = sorted(mode_df["feparam_offset"].dropna().unique().tolist())
        sizes = sorted(mode_df["feparam_size_bytes"].dropna().unique().tolist())

        print(f"{mode}: offsets={offsets}, sizes={sizes}")


def save_feparam_blob_inventory(blob_inventory_df, output_dir):
    """
    Save FeParam blob inventory.

    Parameters
    ----------
    blob_inventory_df : pandas.DataFrame
        Blob inventory.

    output_dir : pathlib.Path
        Output directory.

    Returns
    -------
    pathlib.Path
        Saved CSV path.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / "nb08_feparam_blob_inventory.csv"

    blob_inventory_df.to_csv(output_path, index=False)

    print(f"Saved FeParam blob inventory CSV: {output_path}")

    return output_path

In [10]:
feparam_blob_inventory_df = build_feparam_blob_inventory(PATHS["NATIVE_BATCH_DIR"])
summarize_feparam_blob_inventory(feparam_blob_inventory_df)
feparam_blob_inventory_csv_path = save_feparam_blob_inventory(feparam_blob_inventory_df, PATHS["NB08_REPORTS_DIR"])

feparam_blob_inventory_df

FeParam blob inventory
Rows: 20

blob_extract_ok counts:
blob_extract_ok
True    20
Name: count, dtype: int64

layout_source counts:
layout_source
fallback_pw_size_bc_size_from_vm_bin_size    10
scope04_fallback_pw_size_offset_0            10
Name: count, dtype: int64



,recording_id,mode,vm_bin_size_bytes,feparam_offset,feparam_size_bytes,layout_source,blob_extract_ok,blob_entropy,ascii_string_count,first_strings_preview
0,202606130411060002SMP,BC,35465,4530,30935,fallback_pw_size_bc_size_from_vm_bin_size,True,4.7782,730,"""int32::APowerLevel[1]"" : [ | ""uint8::AngleNum..."
1,202606130411060002SMP,PW,35465,0,4514,scope04_fallback_pw_size_offset_0,True,4.3730,115,"""int32::APowerLevel[1]"" : [ | ""float::AttenuCa..."
2,202606130413540003SMP,BC,35479,4530,30949,fallback_pw_size_bc_size_from_vm_bin_size,True,4.7821,730,"""int32::APowerLevel[1]"" : [ | ""uint8::AngleNum..."
3,202606130413540003SMP,PW,35479,0,4514,scope04_fallback_pw_size_offset_0,True,4.3777,115,"""int32::APowerLevel[1]"" : [ | ""float::AttenuCa..."
4,202606130417060004SMP,BC,35483,4530,30953,fallback_pw_size_bc_size_from_vm_bin_size,True,4.7476,726,":L1_d_iSampleX[1]"" : [ | ""uint32::L1_d_iSample..."
5,202606130417060004SMP,PW,35483,0,4514,scope04_fallback_pw_size_offset_0,True,4.8796,120,"""int32::APowerLevel[1]"" : [ | ""uint8::AngleNum..."
6,202606130420260005SMP,BC,35470,4530,30940,fallback_pw_size_bc_size_from_vm_bin_size,True,4.7801,730,"""int32::APowerLevel[1]"" : [ | ""uint8::AngleNum..."
7,202606130420260005SMP,PW,35470,0,4514,scope04_fallback_pw_size_offset_0,True,4.3746,116,"""int32::APowerLevel[1]"" : [ | ""float::AttenuCa..."
8,202606130422440006SMP,BC,35480,4530,30950,fallback_pw_size_bc_size_from_vm_bin_size,True,4.7478,725,":L1_d_iSampleX[1]"" : [ | ""uint32::L1_d_iSample..."
9,202606130422440006SMP,PW,35480,0,4514,scope04_fallback_pw_size_offset_0,True,4.8796,120,"""int32::APowerLevel[1]"" : [ | ""uint8::AngleNum..."



Unique offsets/sizes:
PW: offsets=[0], sizes=[4514]
BC: offsets=[4530], sizes=[30935, 30939, 30940, 30949, 30950, 30953, 30954, 31080]
Saved FeParam blob inventory CSV: E:\DopplerLab\reports\nb08_feparam\nb08_feparam_blob_inventory.csv


,recording_id,mode,vm_txt_exists,vm_bin_exists,vm_bin_size_bytes,feparam_offset,feparam_size_bytes,layout_source,layout_txt_size,layout_file_consistent_size,layout_source_text,blob_extract_ok,blob_error,blob_entropy,ascii_string_count,first_strings_preview,vm_txt_path,vm_bin_path
0,202606130411060002SMP,BC,True,True,35465,4530,30935,fallback_pw_size_bc_size_from_vm_bin_size,None,30935,expected_at_least_2_param_sizes_found_0,True,,4.7782,730,"""int32::APowerLevel[1]"" : [ | ""uint8::AngleNum...",E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...
1,202606130411060002SMP,PW,True,True,35465,0,4514,scope04_fallback_pw_size_offset_0,None,4514,expected_at_least_2_param_sizes_found_0,True,,4.3730,115,"""int32::APowerLevel[1]"" : [ | ""float::AttenuCa...",E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...
2,202606130413540003SMP,BC,True,True,35479,4530,30949,fallback_pw_size_bc_size_from_vm_bin_size,None,30949,expected_at_least_2_param_sizes_found_0,True,,4.7821,730,"""int32::APowerLevel[1]"" : [ | ""uint8::AngleNum...",E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...
3,202606130413540003SMP,PW,True,True,35479,0,4514,scope04_fallback_pw_size_offset_0,None,4514,expected_at_least_2_param_sizes_found_0,True,,4.3777,115,"""int32::APowerLevel[1]"" : [ | ""float::AttenuCa...",E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...
4,202606130417060004SMP,BC,True,True,35483,4530,30953,fallback_pw_size_bc_size_from_vm_bin_size,None,30953,expected_at_least_2_param_sizes_found_0,True,,4.7476,726,":L1_d_iSampleX[1]"" : [ | ""uint32::L1_d_iSample...",E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...
5,202606130417060004SMP,PW,True,True,35483,0,4514,scope04_fallback_pw_size_offset_0,None,4514,expected_at_least_2_param_sizes_found_0,True,,4.8796,120,"""int32::APowerLevel[1]"" : [ | ""uint8::AngleNum...",E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...
6,202606130420260005SMP,BC,True,True,35470,4530,30940,fallback_pw_size_bc_size_from_vm_bin_size,None,30940,expected_at_least_2_param_sizes_found_0,True,,4.7801,730,"""int32::APowerLevel[1]"" : [ | ""uint8::AngleNum...",E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...
7,202606130420260005SMP,PW,True,True,35470,0,4514,scope04_fallback_pw_size_offset_0,None,4514,expected_at_least_2_param_sizes_found_0,True,,4.3746,116,"""int32::APowerLevel[1]"" : [ | ""float::AttenuCa...",E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...
8,202606130422440006SMP,BC,True,True,35480,4530,30950,fallback_pw_size_bc_size_from_vm_bin_size,None,30950,expected_at_least_2_param_sizes_found_0,True,,4.7478,725,":L1_d_iSampleX[1]"" : [ | ""uint32::L1_d_iSample...",E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...
9,202606130422440006SMP,PW,True,True,35480,0,4514,scope04_fallback_pw_size_offset_0,None,4514,expected_at_least_2_param_sizes_found_0,True,,4.8796,120,"""int32::APowerLevel[1]"" : [ | ""uint8::AngleNum...",E:\DopplerLab\ultrasound_recordings\batch_2026...,E:\DopplerLab\ultrasound_recordings\batch_2026...


## FeParam string and parameter-name inventory

This section extracts readable strings from PW and BC FeParam blobs.

Main goal:

- inventory readable parameter-like strings,
- extract parameter names such as `APowerLevel`, `AttenuCalFreq`, `DFocus`,
- compare presence across recordings and modes,
- identify stable vs variable metadata candidates.

Important:

This section does not decode numeric values yet.
It only builds a parameter-name inventory.

In [11]:
def load_feparam_blob_from_inventory_row(row):
    """
    Load one FeParam blob using a row from feparam_blob_inventory_df.

    Parameters
    ----------
    row : pandas.Series
        One row from FeParam blob inventory.

    Returns
    -------
    bytes
        Extracted FeParam blob.
    """
    vm_bin_path = Path(row["vm_bin_path"])
    offset = int(row["feparam_offset"])
    size = int(row["feparam_size_bytes"])

    return extract_feparam_blob(vm_bin_path=vm_bin_path, offset=offset, size=size)


def normalize_feparam_string(raw_string):
    """
    Normalize a readable string extracted from a FeParam blob.

    Parameters
    ----------
    raw_string : str
        Raw extracted string.

    Returns
    -------
    str
        Cleaned string.
    """
    if raw_string is None:
        return ""

    text = str(raw_string)
    text = text.replace("\x00", "")
    text = text.replace("\r", " ")
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()

    return text


def extract_parameter_signature_from_string(text):
    """
    Extract parameter signature from a FeParam readable string.

    Expected examples:

        "int32::APowerLevel[1]" : [
        "float::AttenuCalFreq[1]" : [
        "uint32::L1_d_iSampleX[1]" : [

    Returns the type and parameter name when a typed parameter-like
    pattern is found.

    Parameters
    ----------
    text : str
        Cleaned FeParam string.

    Returns
    -------
    dict
        Extracted parameter signature fields.
    """
    text = normalize_feparam_string(text)

    # Most useful pattern:
    # "int32::APowerLevel[1]" : [
    pattern = re.compile(
        r'"?(?P<dtype>[A-Za-z0-9_]+)::(?P<name>[A-Za-z_][A-Za-z0-9_]*)'
        r'(?:\[(?P<count>\d+)\])?"?'
    )

    match = pattern.search(text)

    if match is None:
        return {
            "is_parameter_like": False,
            "param_dtype": "",
            "param_name": "",
            "param_count": None,
        }

    count_value = match.group("count")

    return {
        "is_parameter_like": True,
        "param_dtype": match.group("dtype"),
        "param_name": match.group("name"),
        "param_count": int(count_value) if count_value is not None else None,
    }


def build_feparam_string_inventory(feparam_blob_inventory_df, min_length=4):
    """
    Build readable string inventory for all FeParam blobs.

    Parameters
    ----------
    feparam_blob_inventory_df : pandas.DataFrame
        FeParam blob inventory from previous cell.

    min_length : int
        Minimum readable ASCII string length.

    Returns
    -------
    pandas.DataFrame
        One row per extracted readable string.
    """
    rows = []

    for _, blob_row in feparam_blob_inventory_df.iterrows():
        recording_id = blob_row["recording_id"]
        mode = blob_row["mode"]

        if not bool(blob_row["blob_extract_ok"]):
            continue

        blob = load_feparam_blob_from_inventory_row(blob_row)

        strings = extract_ascii_strings(
            blob,
            min_length=min_length,
        )

        for string_index, raw_string in enumerate(strings):
            clean_string = normalize_feparam_string(raw_string)
            signature = extract_parameter_signature_from_string(clean_string)

            rows.append(
                {
                    "recording_id": recording_id,
                    "mode": mode,
                    "string_index": string_index,
                    "string_text": clean_string,
                    "string_length": len(clean_string),
                    "is_parameter_like": signature["is_parameter_like"],
                    "param_dtype": signature["param_dtype"],
                    "param_name": signature["param_name"],
                    "param_count": signature["param_count"],
                }
            )

    string_inventory_df = pd.DataFrame(rows)

    if len(string_inventory_df) == 0:
        return string_inventory_df

    return string_inventory_df.sort_values(["recording_id", "mode", "string_index"]).reset_index(drop=True)


def build_feparam_parameter_presence_summary(string_inventory_df):
    """
    Build parameter presence summary by mode and parameter name.

    Parameters
    ----------
    string_inventory_df : pandas.DataFrame
        String inventory table.

    Returns
    -------
    pandas.DataFrame
        Parameter presence summary.
    """
    parameter_df = string_inventory_df[string_inventory_df["is_parameter_like"]].copy()

    if len(parameter_df) == 0:
        return pd.DataFrame()

    rows = []

    for (mode, param_name, param_dtype), group_df in parameter_df.groupby(["mode", "param_name", "param_dtype"]):
        recording_ids = sorted(group_df["recording_id"].unique().tolist())
        n_recordings = len(recording_ids)
        n_occurrences = len(group_df)

        string_examples = (group_df["string_text"].drop_duplicates().head(5).tolist())
        rows.append(
            {
                "mode": mode,
                "param_name": param_name,
                "param_dtype": param_dtype,
                "n_recordings": n_recordings,
                "n_occurrences": n_occurrences,
                "recording_ids": ";".join(recording_ids),
                "example_strings": " | ".join(string_examples),
            }
        )

    summary_df = pd.DataFrame(rows)
    summary_df = summary_df.sort_values(["mode", "n_recordings", "param_name"], ascending=[True, False, True],).reset_index(drop=True)

    return summary_df


def build_feparam_parameter_matrix(string_inventory_df):
    """
    Build a recording x parameter presence matrix.

    Parameters
    ----------
    string_inventory_df : pandas.DataFrame
        String inventory table.

    Returns
    -------
    pandas.DataFrame
        Presence matrix with one row per recording/mode.
    """
    parameter_df = string_inventory_df[string_inventory_df["is_parameter_like"]].copy()

    if len(parameter_df) == 0:
        return pd.DataFrame()

    parameter_df["param_key"] = (
        parameter_df["mode"].astype(str)
        + "::"
        + parameter_df["param_dtype"].astype(str)
        + "::"
        + parameter_df["param_name"].astype(str)
    )

    presence_df = (
        parameter_df
        .assign(present=True)
        .pivot_table(
            index=["recording_id", "mode"],
            columns="param_key",
            values="present",
            aggfunc="max",
            fill_value=False,
        )
        .reset_index()
    )

    presence_df.columns.name = None
    return presence_df


def summarize_feparam_string_inventory(string_inventory_df, parameter_summary_df,):
    """
    Print compact summary of FeParam string and parameter inventory.

    Parameters
    ----------
    string_inventory_df : pandas.DataFrame
        String inventory table.

    parameter_summary_df : pandas.DataFrame
        Parameter presence summary.
    """
    print("FeParam string inventory")
    print("=" * 80)
    print(f"Readable strings: {len(string_inventory_df)}")

    if len(string_inventory_df) == 0:
        return

    parameter_like_count = int(string_inventory_df["is_parameter_like"].sum())

    print(f"Parameter-like strings: {parameter_like_count}")
    print()
    print("Readable string count by mode:")
    print(string_inventory_df.groupby("mode").size())
    print()
    print("Parameter-like count by mode:")
    print(string_inventory_df[string_inventory_df["is_parameter_like"]].groupby("mode").size())
    print()

    print("Unique parameter names by mode:")
    if len(parameter_summary_df) > 0:
        print(parameter_summary_df.groupby("mode")["param_name"].nunique())
    print()

    print("Top parameter names by mode/presence:")
    display(parameter_summary_df.head(40))

    print()
    print("Example non-parameter readable strings:")
    non_param_examples = (string_inventory_df[~string_inventory_df["is_parameter_like"]]["string_text"].drop_duplicates().head(20).tolist())

    for item in non_param_examples:
        print(f"  - {item}")


def save_feparam_string_outputs(
    string_inventory_df,
    parameter_summary_df,
    parameter_matrix_df,
    output_dir,
):
    """
    Save FeParam string inventory outputs.

    Parameters
    ----------
    string_inventory_df : pandas.DataFrame
        String inventory table.

    parameter_summary_df : pandas.DataFrame
        Parameter summary table.

    parameter_matrix_df : pandas.DataFrame
        Parameter presence matrix.

    output_dir : pathlib.Path
        Output directory.

    Returns
    -------
    dict
        Saved output paths.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    string_inventory_path = output_dir / "nb08_feparam_string_inventory.csv"
    parameter_summary_path = output_dir / "nb08_feparam_parameter_name_summary.csv"
    parameter_matrix_path = output_dir / "nb08_feparam_parameter_presence_matrix.csv"
    string_inventory_df.to_csv(string_inventory_path, index=False)
    parameter_summary_df.to_csv(parameter_summary_path, index=False)
    parameter_matrix_df.to_csv(parameter_matrix_path, index=False)
    print(f"Saved string inventory CSV: {string_inventory_path}")
    print(f"Saved parameter summary CSV: {parameter_summary_path}")
    print(f"Saved parameter presence matrix CSV: {parameter_matrix_path}")

    return {
        "string_inventory_csv": string_inventory_path,
        "parameter_summary_csv": parameter_summary_path,
        "parameter_presence_matrix_csv": parameter_matrix_path,
    }

In [12]:
feparam_string_inventory_df = build_feparam_string_inventory(feparam_blob_inventory_df, min_length=4,)
feparam_parameter_summary_df = build_feparam_parameter_presence_summary(feparam_string_inventory_df)
feparam_parameter_matrix_df = build_feparam_parameter_matrix(feparam_string_inventory_df)
summarize_feparam_string_inventory(string_inventory_df=feparam_string_inventory_df, parameter_summary_df=feparam_parameter_summary_df,)

feparam_string_output_paths = save_feparam_string_outputs(
    string_inventory_df=feparam_string_inventory_df,
    parameter_summary_df=feparam_parameter_summary_df,
    parameter_matrix_df=feparam_parameter_matrix_df,
    output_dir=PATHS["NB08_REPORTS_DIR"],
)

feparam_parameter_summary_df

FeParam string inventory
Readable strings: 8467
Parameter-like strings: 6787

Readable string count by mode:
mode
BC    7300
PW    1167
dtype: int64

Parameter-like count by mode:
mode
BC    6143
PW     644
dtype: int64

Unique parameter names by mode:
mode
BC    584
PW    143
Name: param_name, dtype: int64

Top parameter names by mode/presence:


,mode,param_name,param_dtype,n_recordings,n_occurrences,recording_ids,example_strings
0,BC,APowerLevel,int32,10,10,202606130411060002SMP;202606130413540003SMP;20...,"""int32::APowerLevel[1]"" : ["
1,BC,Angle,float,10,150,202606130411060002SMP;202606130413540003SMP;20...,"""float::Angle[1]"" : ["
2,BC,BiTouchBirghtLevel,int32,10,10,202606130411060002SMP;202606130413540003SMP;20...,"""int32::BiTouchBirghtLevel[1]"" : ["
3,BC,BiTouchLevel,int32,10,10,202606130411060002SMP;202606130413540003SMP;20...,"""int32::BiTouchLevel[1]"" : ["
4,BC,BiTouchSwitch,bool,10,10,202606130411060002SMP;202606130413540003SMP;20...,"""bool::BiTouchSwitch[1]"" : ["
5,BC,BlindDistance,float,10,150,202606130411060002SMP;202606130413540003SMP;20...,"""float::BlindDistance[1]"" : ["
6,BC,CAdaptB,float,10,10,202606130411060002SMP;202606130413540003SMP;20...,"""float::CAdaptB[1]"" : ["
7,BC,CAdaptK,float,10,10,202606130411060002SMP;202606130413540003SMP;20...,"""float::CAdaptK[1]"" : ["
8,BC,CAdaptPersistenceOn,bool,10,10,202606130411060002SMP;202606130413540003SMP;20...,"""bool::CAdaptPersistenceOn[1]"" : ["
9,BC,CDSPMethod,uint8,10,10,202606130411060002SMP;202606130413540003SMP;20...,"""uint8::CDSPMethod[1]"" : ["



Example non-parameter readable strings:
  - false
  - 126,
  - 0.1331010472795274
  - -19.004501342773438,
  - 19.004501342773438
  - -144,
  - true
  - 16.5,
  - 16.5
  - 101,
  - 115,
  - 0.0038900000508875
  - 32768,
  - 41.087673187255859
  - 0.0577500015497208
  - -9.5022506713867187,
  - 9.5022506713867187
  - 8.75,
  - 26.25
  - "BiClearPara[0]" : {
Saved string inventory CSV: E:\DopplerLab\reports\nb08_feparam\nb08_feparam_string_inventory.csv
Saved parameter summary CSV: E:\DopplerLab\reports\nb08_feparam\nb08_feparam_parameter_name_summary.csv
Saved parameter presence matrix CSV: E:\DopplerLab\reports\nb08_feparam\nb08_feparam_parameter_presence_matrix.csv


,mode,param_name,param_dtype,n_recordings,n_occurrences,recording_ids,example_strings
0,BC,APowerLevel,int32,10,10,202606130411060002SMP;202606130413540003SMP;20...,"""int32::APowerLevel[1]"" : ["
1,BC,Angle,float,10,150,202606130411060002SMP;202606130413540003SMP;20...,"""float::Angle[1]"" : ["
2,BC,BiTouchBirghtLevel,int32,10,10,202606130411060002SMP;202606130413540003SMP;20...,"""int32::BiTouchBirghtLevel[1]"" : ["
3,BC,BiTouchLevel,int32,10,10,202606130411060002SMP;202606130413540003SMP;20...,"""int32::BiTouchLevel[1]"" : ["
4,BC,BiTouchSwitch,bool,10,10,202606130411060002SMP;202606130413540003SMP;20...,"""bool::BiTouchSwitch[1]"" : ["
...,...,...,...,...,...,...,...
722,PW,L1_m_iSampleY,uint32,3,3,202606130417060004SMP;202606130422440006SMP;20...,"""uint32::L1_m_iSampleY[1]"" : ["
723,PW,L1_m_lo,float,3,3,202606130417060004SMP;202606130422440006SMP;20...,"""float::L1_m_lo[1]"" : ["
724,PW,L1_m_slp,float,3,3,202606130417060004SMP;202606130422440006SMP;20...,"""float::L1_m_slp[1]"" : ["
725,PW,MultiScale_Enable,uint32,3,3,202606130417060004SMP;202606130422440006SMP;20...,"""uint32::MultiScale_Enable[1]"" : ["


## Experimental FeParam value extraction

This section tries to extract raw values following parameter-like strings.

The parser is intentionally conservative:

- it scans readable text tokens,
- detects typed parameter headers such as `"int32::APowerLevel[1]" : [`,
- captures the following value tokens until the closing bracket or next parameter header,
- stores raw values without clinical interpretation.

Important:

This is experimental metadata extraction.

A value may represent a machine setting, internal setting, array element, lookup table entry,
or nested structure. No clinical meaning is assigned here.

In [13]:
def extract_printable_tokens(byte_data, min_length=1):
    """
    Extract printable ASCII tokens from binary data.

    This is similar to extract_ascii_strings(), but allows very short tokens
    so that numeric values such as '0', '1', ']', '[' are not lost.

    Parameters
    ----------
    byte_data : bytes
        Binary blob.

    min_length : int
        Minimum token length.

    Returns
    -------
    list[str]
        Printable tokens.
    """
    if byte_data is None:
        return []

    pattern = rb"[ -~]{" + str(min_length).encode("ascii") + rb",}"
    matches = re.findall(pattern, byte_data)
    tokens = [match.decode("ascii", errors="replace") for match in matches]
    tokens = [normalize_feparam_string(token) for token in tokens]
    tokens = [token for token in tokens if token != ""]

    return tokens


def token_is_parameter_header(token):
    """
    Check whether a token looks like a typed FeParam parameter header.

    Parameters
    ----------
    token : str
        Printable token.

    Returns
    -------
    bool
        True if token is parameter-like.
    """
    signature = extract_parameter_signature_from_string(token)
    
    return bool(signature["is_parameter_like"])


def clean_value_token(token):
    """
    Clean a raw value token extracted from FeParam text.

    Parameters
    ----------
    token : str
        Raw token.

    Returns
    -------
    str
        Clean value-like token.
    """
    token = normalize_feparam_string(token)
    # Remove common JSON-ish punctuation around scalar values.
    token = token.strip()
    token = token.rstrip(",")

    return token


def parse_scalar_value_token(token):
    """
    Parse a scalar value token into bool/int/float/string.

    Parameters
    ----------
    token : str
        Clean value token.

    Returns
    -------
    tuple[object, str]
        Parsed value and parsed type label.
    """
    token = clean_value_token(token)

    if token == "":
        return None, "empty"

    lower = token.lower()

    if lower == "true":
        return True, "bool"
    if lower == "false":
        return False, "bool"

    # Remove surrounding quotes for plain strings.
    if len(token) >= 2 and token[0] == '"' and token[-1] == '"':
        return token[1:-1], "string"
        
    # Integer first, then float.
    if re.fullmatch(r"[-+]?\d+", token):
        try:
            return int(token), "int"
        except Exception:
            pass

    if re.fullmatch(r"[-+]?(?:\d+\.\d*|\d*\.\d+|\d+)(?:[eE][-+]?\d+)?", token):
        try:
            return float(token), "float"
        except Exception:
            pass

    return token, "string"


def collect_value_tokens_after_header(tokens, header_index, max_tokens=500):
    """
    Collect raw value tokens after a parameter header.

    The parser starts after the header token and stops at:
    - a standalone closing bracket,
    - the next parameter header,
    - max_tokens safety limit.

    Parameters
    ----------
    tokens : list[str]
        Printable token list.

    header_index : int
        Index of parameter header token.

    max_tokens : int
        Safety limit.

    Returns
    -------
    list[str]
        Raw value tokens.
    """
    value_tokens = []

    for idx in range(header_index + 1, min(len(tokens), header_index + 1 + max_tokens)):
        token = tokens[idx]

        if token_is_parameter_header(token):
            break

        cleaned = clean_value_token(token)

        if cleaned in ["[", "{", ""]:
            continue

        if cleaned in ["]", "}", "],", "},"]:
            break

        # Stop at object/block headers. These are not scalar parameter values.
        if cleaned.endswith("{") or cleaned.endswith("["):
            # Keep named nested structures out of scalar extraction.
            continue

        value_tokens.append(cleaned)

        # Many scalar params are one-line values followed by ']'.
        # We do not stop immediately because arrays are common.

    return value_tokens


def classify_extracted_value_list(parsed_values):
    """
    Classify extracted value list.

    Parameters
    ----------
    parsed_values : list[object]
        Parsed values.

    Returns
    -------
    str
        Value class label.
    """
    if len(parsed_values) == 0:
        return "empty"
    if all(isinstance(value, bool) for value in parsed_values):
        return "bool_array" if len(parsed_values) > 1 else "bool_scalar"
    if all(isinstance(value, int) and not isinstance(value, bool) for value in parsed_values):
        return "int_array" if len(parsed_values) > 1 else "int_scalar"

    if all(isinstance(value, (int, float)) and not isinstance(value, bool) for value in parsed_values):
        return "numeric_array" if len(parsed_values) > 1 else "numeric_scalar"

    if all(isinstance(value, str) for value in parsed_values):
        return "string_array" if len(parsed_values) > 1 else "string_scalar"

    return "mixed"


def build_feparam_raw_value_table(feparam_blob_inventory_df):
    """
    Build experimental raw value extraction table from FeParam blobs.

    Parameters
    ----------
    feparam_blob_inventory_df : pandas.DataFrame
        FeParam blob inventory.

    Returns
    -------
    pandas.DataFrame
        One row per parameter occurrence.
    """
    rows = []

    for _, blob_row in feparam_blob_inventory_df.iterrows():
        recording_id = blob_row["recording_id"]
        mode = blob_row["mode"]

        if not bool(blob_row["blob_extract_ok"]):
            continue

        blob = load_feparam_blob_from_inventory_row(blob_row)
        tokens = extract_printable_tokens(blob, min_length=1)
        occurrence_counter = defaultdict(int)

        for token_index, token in enumerate(tokens):
            signature = extract_parameter_signature_from_string(token)

            if not signature["is_parameter_like"]:
                continue

            param_dtype = signature["param_dtype"]
            param_name = signature["param_name"]
            param_count = signature["param_count"]
            occurrence_key = (mode, param_dtype, param_name)
            occurrence_counter[occurrence_key] += 1
            occurrence_index = occurrence_counter[occurrence_key]
            raw_value_tokens = collect_value_tokens_after_header(tokens=tokens, header_index=token_index, max_tokens=500)
            
            parsed_values = []
            parsed_types = []

            for raw_value in raw_value_tokens:
                parsed_value, parsed_type = parse_scalar_value_token(raw_value)
                parsed_values.append(parsed_value)
                parsed_types.append(parsed_type)

            value_class = classify_extracted_value_list(parsed_values)

            # Keep values as strings for CSV stability.
            raw_values_joined = "|".join(raw_value_tokens)

            parsed_values_joined = "|".join([str(value) for value in parsed_values])
            unique_parsed_types = ";".join(sorted(set(parsed_types)))

            rows.append(
                {
                    "recording_id": recording_id,
                    "mode": mode,
                    "token_index": token_index,
                    "param_dtype": param_dtype,
                    "param_name": param_name,
                    "param_count_declared": param_count,
                    "occurrence_index_within_recording_mode": occurrence_index,
                    "n_raw_value_tokens": len(raw_value_tokens),
                    "value_class": value_class,
                    "parsed_value_types": unique_parsed_types,
                    "raw_values": raw_values_joined,
                    "parsed_values": parsed_values_joined,
                    "header_token": token,
                }
            )

    value_df = pd.DataFrame(rows)
    if len(value_df) == 0:
        return value_df

    value_df = value_df.sort_values(["recording_id", "mode", "token_index",]).reset_index(drop=True)

    return value_df


def build_feparam_value_stability_summary(value_df):
    """
    Summarize extracted raw values across recordings.

    Parameters
    ----------
    value_df : pandas.DataFrame
        Raw value table.

    Returns
    -------
    pandas.DataFrame
        Value stability summary by mode/parameter/occurrence.
    """
    if len(value_df) == 0:
        return pd.DataFrame()

    rows = []
    group_cols = ["mode", "param_dtype", "param_name", "occurrence_index_within_recording_mode",]

    for group_key, group_df in value_df.groupby(group_cols):
        mode, param_dtype, param_name, occurrence_index = group_key

        recording_ids = sorted(group_df["recording_id"].unique().tolist())
        n_recordings = len(recording_ids)
        parsed_values_unique = sorted(group_df["parsed_values"].fillna("").astype(str).unique().tolist())

        raw_values_unique = sorted(group_df["raw_values"].fillna("").astype(str).unique().tolist())
        value_classes = sorted(group_df["value_class"].fillna("").astype(str).unique().tolist())
        n_unique_parsed_values = len(parsed_values_unique)

        if n_recordings == 10 and n_unique_parsed_values == 1:
            stability_label = "stable_10_of_10"
        elif n_recordings == 10 and n_unique_parsed_values > 1:
            stability_label = "variable_10_of_10"
        elif n_recordings < 10 and n_unique_parsed_values == 1:
            stability_label = "stable_subset"
        else:
            stability_label = "variable_subset"

        rows.append(
            {
                "mode": mode,
                "param_dtype": param_dtype,
                "param_name": param_name,
                "occurrence_index_within_recording_mode": occurrence_index,
                "n_recordings": n_recordings,
                "n_unique_parsed_values": n_unique_parsed_values,
                "stability_label": stability_label,
                "value_classes": ";".join(value_classes),
                "recording_ids": ";".join(recording_ids),
                "example_parsed_values": " || ".join(parsed_values_unique[:5]),
                "example_raw_values": " || ".join(raw_values_unique[:5]),
            }
        )

    summary_df = pd.DataFrame(rows)
    summary_df = summary_df.sort_values(
        [
            "mode",
            "stability_label",
            "n_recordings",
            "param_name",
            "occurrence_index_within_recording_mode",
        ],
        ascending=[True, True, False, True, True],
    ).reset_index(drop=True)

    return summary_df


def summarize_feparam_value_extraction(value_df, value_summary_df):
    """
    Print compact summary of FeParam value extraction.

    Parameters
    ----------
    value_df : pandas.DataFrame
        Raw value extraction table.

    value_summary_df : pandas.DataFrame
        Value stability summary.
    """
    print("FeParam raw value extraction")
    print("=" * 80)
    print(f"Parameter occurrences extracted: {len(value_df)}")
    print()

    if len(value_df) == 0:
        return

    print("Value class counts:")
    print(value_df["value_class"].value_counts(dropna=False))
    print()
    print("Occurrences by mode:")
    print(value_df.groupby("mode").size())
    print()
    print("Stability labels:")
    print(value_summary_df["stability_label"].value_counts(dropna=False))
    print()
    print("Stable 10/10 examples:")
    stable_examples = value_summary_df[value_summary_df["stability_label"] == "stable_10_of_10"].head(30)

    display(stable_examples)

    print()
    print("Variable 10/10 examples:")
    variable_examples = value_summary_df[value_summary_df["stability_label"] == "variable_10_of_10"].head(30)
    display(variable_examples)

    print()
    print("Raw extraction examples:")
    display(
        value_df[
            [
                "recording_id",
                "mode",
                "param_dtype",
                "param_name",
                "occurrence_index_within_recording_mode",
                "param_count_declared",
                "n_raw_value_tokens",
                "value_class",
                "parsed_values",
            ]
        ].head(40)
    )


def save_feparam_value_outputs(value_df, value_summary_df, output_dir,):
    """
    Save FeParam raw value extraction outputs.

    Parameters
    ----------
    value_df : pandas.DataFrame
        Raw value table.

    value_summary_df : pandas.DataFrame
        Value stability summary.

    output_dir : pathlib.Path
        Output directory.

    Returns
    -------
    dict
        Saved paths.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    value_table_path = output_dir / "nb08_feparam_raw_value_table.csv"
    value_summary_path = output_dir / "nb08_feparam_value_stability_summary.csv"
    value_df.to_csv(value_table_path, index=False)
    value_summary_df.to_csv(value_summary_path, index=False)

    print(f"Saved raw value table CSV: {value_table_path}")
    print(f"Saved value stability summary CSV: {value_summary_path}")

    return {
        "raw_value_table_csv": value_table_path,
        "value_stability_summary_csv": value_summary_path,
    }

In [14]:
feparam_raw_value_df = build_feparam_raw_value_table(feparam_blob_inventory_df)
feparam_value_summary_df = build_feparam_value_stability_summary(feparam_raw_value_df)
summarize_feparam_value_extraction(value_df=feparam_raw_value_df, value_summary_df=feparam_value_summary_df,)
feparam_value_output_paths = save_feparam_value_outputs(
    value_df=feparam_raw_value_df,
    value_summary_df=feparam_value_summary_df,
    output_dir=PATHS["NB08_REPORTS_DIR"],
)
feparam_value_summary_df

FeParam raw value extraction
Parameter occurrences extracted: 6787

Value class counts:
value_class
int_scalar        5293
int_array          679
bool_scalar        340
numeric_scalar     320
numeric_array      101
empty               34
bool_array          20
Name: count, dtype: int64

Occurrences by mode:
mode
BC    6143
PW     644
dtype: int64

Stability labels:
stability_label
stable_10_of_10      496
stable_subset        266
variable_10_of_10     40
variable_subset       19
Name: count, dtype: int64

Stable 10/10 examples:


,mode,param_dtype,param_name,occurrence_index_within_recording_mode,n_recordings,n_unique_parsed_values,stability_label,value_classes,recording_ids,example_parsed_values,example_raw_values
0,BC,int32,APowerLevel,1,10,1,stable_10_of_10,int_scalar,202606130411060002SMP;202606130413540003SMP;20...,99,99
1,BC,float,Angle,1,10,1,stable_10_of_10,numeric_scalar,202606130411060002SMP;202606130413540003SMP;20...,-0.2094399929046631,-0.2094399929046631
2,BC,float,Angle,2,10,1,stable_10_of_10,int_scalar,202606130411060002SMP;202606130413540003SMP;20...,0,0
3,BC,float,Angle,3,10,1,stable_10_of_10,numeric_scalar,202606130411060002SMP;202606130413540003SMP;20...,0.2094399929046631,0.2094399929046631
4,BC,float,Angle,4,10,1,stable_10_of_10,int_scalar,202606130411060002SMP;202606130413540003SMP;20...,0,0
5,BC,float,Angle,5,10,1,stable_10_of_10,int_scalar,202606130411060002SMP;202606130413540003SMP;20...,0,0
6,BC,float,Angle,6,10,1,stable_10_of_10,int_scalar,202606130411060002SMP;202606130413540003SMP;20...,0,0
7,BC,float,Angle,7,10,1,stable_10_of_10,int_scalar,202606130411060002SMP;202606130413540003SMP;20...,0,0
8,BC,float,Angle,8,10,1,stable_10_of_10,int_scalar,202606130411060002SMP;202606130413540003SMP;20...,0,0
9,BC,float,Angle,9,10,1,stable_10_of_10,int_scalar,202606130411060002SMP;202606130413540003SMP;20...,0,0



Variable 10/10 examples:


,mode,param_dtype,param_name,occurrence_index_within_recording_mode,n_recordings,n_unique_parsed_values,stability_label,value_classes,recording_ids,example_parsed_values,example_raw_values
633,BC,float,CAdaptB,1,10,2,variable_10_of_10,int_scalar,202606130411060002SMP;202606130413540003SMP;20...,0 || 1,0 || 1
634,BC,float,CAdaptK,1,10,2,variable_10_of_10,int_scalar,202606130411060002SMP;202606130413540003SMP;20...,-2 || 0,-2 || 0
635,BC,bool,CAdaptPersistenceOn,1,10,2,variable_10_of_10,bool_scalar,202606130411060002SMP;202606130413540003SMP;20...,False || True,false || true
636,BC,int8,CDispFreq,1,10,2,variable_10_of_10,int_array,202606130411060002SMP;202606130413540003SMP;20...,0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0 || 53|...,0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0 || 53|...
637,BC,double,CDispLineGap,1,10,2,variable_10_of_10,int_scalar;numeric_scalar,202606130411060002SMP;202606130413540003SMP;20...,0 || 0.1598326383649555,0 || 0.1598326383649555
638,BC,uint8,CFrameCompCoef,1,10,2,variable_10_of_10,int_array,202606130411060002SMP;202606130413540003SMP;20...,0|0|0 || 128|128|128,0|0|0 || 128|128|128
639,BC,bool,CFrameCompOn,1,10,2,variable_10_of_10,bool_scalar,202606130411060002SMP;202606130413540003SMP;20...,False || True,false || true
640,BC,int32,CGainLevel,1,10,2,variable_10_of_10,int_scalar,202606130411060002SMP;202606130413540003SMP;20...,25 || 29,25 || 29
641,BC,uint32,CKeyholeMagn_Thre,1,10,2,variable_10_of_10,int_scalar,202606130411060002SMP;202606130413540003SMP;20...,0 || 4000,0 || 4000
642,BC,int8,CKeyholeMinVel_Thre,1,10,2,variable_10_of_10,int_scalar,202606130411060002SMP;202606130413540003SMP;20...,0 || 37,0 || 37



Raw extraction examples:


,recording_id,mode,param_dtype,param_name,occurrence_index_within_recording_mode,param_count_declared,n_raw_value_tokens,value_class,parsed_values
0,202606130411060002SMP,BC,int32,APowerLevel,1,1.0,1,int_scalar,99
1,202606130411060002SMP,BC,uint8,AngleNumValid,1,1.0,1,int_scalar,3
2,202606130411060002SMP,BC,bool,BCSameWidth,1,1.0,1,bool_scalar,False
3,202606130411060002SMP,BC,int32,BDRLevel,1,1.0,1,int_scalar,30
4,202606130411060002SMP,BC,float,BDeflectAngle,1,1.0,1,int_scalar,0
5,202606130411060002SMP,BC,int32,BDeflectLevel,1,1.0,1,int_scalar,2
6,202606130411060002SMP,BC,int8,BDispFreq,1,20.0,20,int_array,55|46|54|126|49|54|46|48|0|0|0|0|0|0|0|0|0|0|0|0
7,202606130411060002SMP,BC,double,BDispLineGap,1,1.0,1,numeric_scalar,0.1331010472795274
8,202606130411060002SMP,BC,float,BDispLineRange,1,2.0,2,numeric_array,-19.004501342773438|19.004501342773438
9,202606130411060002SMP,BC,int32,BDispOptLevel,1,1.0,1,int_scalar,2


Saved raw value table CSV: E:\DopplerLab\reports\nb08_feparam\nb08_feparam_raw_value_table.csv
Saved value stability summary CSV: E:\DopplerLab\reports\nb08_feparam\nb08_feparam_value_stability_summary.csv


,mode,param_dtype,param_name,occurrence_index_within_recording_mode,n_recordings,n_unique_parsed_values,stability_label,value_classes,recording_ids,example_parsed_values,example_raw_values
0,BC,int32,APowerLevel,1,10,1,stable_10_of_10,int_scalar,202606130411060002SMP;202606130413540003SMP;20...,99,99
1,BC,float,Angle,1,10,1,stable_10_of_10,numeric_scalar,202606130411060002SMP;202606130413540003SMP;20...,-0.2094399929046631,-0.2094399929046631
2,BC,float,Angle,2,10,1,stable_10_of_10,int_scalar,202606130411060002SMP;202606130413540003SMP;20...,0,0
3,BC,float,Angle,3,10,1,stable_10_of_10,numeric_scalar,202606130411060002SMP;202606130413540003SMP;20...,0.2094399929046631,0.2094399929046631
4,BC,float,Angle,4,10,1,stable_10_of_10,int_scalar,202606130411060002SMP;202606130413540003SMP;20...,0,0
...,...,...,...,...,...,...,...,...,...,...,...
816,PW,int16,BDscLineRange,1,3,2,variable_subset,int_array,202606130417060004SMP;202606130422440006SMP;20...,-120|119 || -144|143,-120|119 || -144|143
817,PW,double,BMaxLineCount,1,3,2,variable_subset,int_scalar,202606130417060004SMP;202606130422440006SMP;20...,240 || 288,240 || 288
818,PW,float,BPersistenceCoef,1,3,2,variable_subset,int_scalar,202606130417060004SMP;202606130422440006SMP;20...,157 || 83,157 || 83
819,PW,float,BUploadFrmRate,1,3,2,variable_subset,numeric_scalar,202606130417060004SMP;202606130422440006SMP;20...,17.738862991333008 || 41.08767318725586,17.738862991333008 || 41.087673187255859


## Candidate acquisition metadata selection

This section selects FeParam entries that look like acquisition metadata candidates.

The goal is to create a smaller table from the large raw value extraction output.

Candidate families include:

- power,
- gain,
- focus,
- frequency / PRF,
- angle / steering,
- depth / range / geometry,
- frame rate / persistence,
- image mode / optimization / dynamic range,
- thresholds / flow-processing settings.

Important:

These are metadata candidates, not validated clinical features.

A parameter can be useful for comparing recordings even if its exact engineering meaning is not fully decoded.

In [15]:
def assign_feparam_candidate_family(param_name):
    """
    Assign a broad acquisition-metadata candidate family from a parameter name.

    This is a heuristic grouping for exploration.
    It does not assign clinical meaning.

    Parameters
    ----------
    param_name : str
        FeParam parameter name.

    Returns
    -------
    str
        Candidate family label.
    """
    name = str(param_name)

    rules = [
        (
            "power",
            [
                "Power",
                "APower",
            ],
        ),
        (
            "gain_tgc_lgc",
            [
                "Gain",
                "TGC",
                "LGC",
            ],
        ),
        (
            "focus",
            [
                "Focus",
            ],
        ),
        (
            "frequency_prf",
            [
                "Freq",
                "Prf",
                "PRF",
                "TxFreq",
                "DispFreq",
            ],
        ),
        (
            "angle_steering",
            [
                "Angle",
                "Deflect",
                "Steer",
            ],
        ),
        (
            "depth_range_geometry",
            [
                "Depth",
                "Range",
                "LineRange",
                "PointRange",
                "LineCount",
                "LineGap",
                "SampleX",
                "SampleY",
                "UploadLine",
                "UploadPoint",
                "DscLine",
                "DscPoint",
            ],
        ),
        (
            "frame_rate_persistence",
            [
                "FrmRate",
                "FrameRate",
                "Frame",
                "Persist",
                "Persistence",
            ],
        ),
        (
            "mode_optimization_dynamic_range",
            [
                "Mode",
                "SubMode",
                "OptLevel",
                "DRLevel",
                "HDR",
                "Scan",
                "Zoom",
                "ImageMode",
                "Density",
            ],
        ),
        (
            "threshold_flow_processing",
            [
                "Thre",
                "Thresh",
                "Vel",
                "WF",
                "Wall",
                "Prior",
                "Flow",
                "Cm_",
                "CWF",
                "NDR",
            ],
        ),
    ]

    for family, keywords in rules:
        for keyword in keywords:
            if keyword.lower() in name.lower():
                return family

    return "other"


def split_value_string(value_string):
    """
    Split a pipe-separated parsed_values/raw_values string.

    Parameters
    ----------
    value_string : object
        Value string.

    Returns
    -------
    list[str]
        Split values.
    """
    if value_string is None:
        return []

    try:
        if pd.isna(value_string):
            return []
    except Exception:
        pass

    text = str(value_string).strip()

    if text == "":
        return []

    return [part.strip() for part in text.split("|")]


def value_token_to_float_or_none(token):
    """
    Convert a token to float if possible.

    Parameters
    ----------
    token : str
        Value token.

    Returns
    -------
    float or None
        Parsed float or None.
    """
    token = str(token).strip()

    if token.lower() in ["true", "false"]:
        return 1.0 if token.lower() == "true" else 0.0

    try:
        return float(token)
    except Exception:
        return None


def classify_zero_nonzero_pattern(parsed_values):
    """
    Classify whether parsed values are zero/false, nonzero/true, mixed, or non-numeric.

    Parameters
    ----------
    parsed_values : str
        Pipe-separated parsed values.

    Returns
    -------
    str
        Pattern label.
    """
    tokens = split_value_string(parsed_values)

    if len(tokens) == 0:
        return "empty"

    numeric_values = [value_token_to_float_or_none(token) for token in tokens]

    if any(value is None for value in numeric_values):
        return "non_numeric_or_mixed"

    abs_values = [abs(value) for value in numeric_values]

    if all(value == 0 for value in abs_values):
        return "all_zero_or_false"

    if all(value > 0 for value in abs_values):
        return "all_nonzero_or_true"

    return "mixed_zero_nonzero"


def build_feparam_candidate_metadata_long(value_df, value_summary_df):
    """
    Build long-form candidate acquisition metadata table.

    Parameters
    ----------
    value_df : pandas.DataFrame
        Raw FeParam value table.

    value_summary_df : pandas.DataFrame
        FeParam value stability summary.

    Returns
    -------
    pandas.DataFrame
        Candidate metadata table, one row per recording/parameter occurrence.
    """
    candidate_df = value_df.copy()
    candidate_df["candidate_family"] = candidate_df["param_name"].apply(assign_feparam_candidate_family)
    candidate_df = candidate_df[candidate_df["candidate_family"] != "other"].copy()

    merge_cols = [
        "mode",
        "param_dtype",
        "param_name",
        "occurrence_index_within_recording_mode",
    ]
    summary_keep_cols = merge_cols + [
        "n_recordings",
        "n_unique_parsed_values",
        "stability_label",
        "example_parsed_values",
    ]

    summary_keep_cols = [col for col in summary_keep_cols if col in value_summary_df.columns]
    candidate_df = candidate_df.merge(value_summary_df[summary_keep_cols], how="left", on=merge_cols,)
    candidate_df["zero_nonzero_pattern"] = candidate_df["parsed_values"].apply(classify_zero_nonzero_pattern)

    candidate_df["candidate_key"] = (
        candidate_df["mode"].astype(str)
        + "__"
        + candidate_df["param_dtype"].astype(str)
        + "__"
        + candidate_df["param_name"].astype(str)
        + "__occ"
        + candidate_df["occurrence_index_within_recording_mode"].astype(str)
    )

    candidate_df["extraction_status"] = "experimental_raw_text_token_extraction"

    preferred_cols = [
        "recording_id",
        "mode",
        "candidate_family",
        "candidate_key",
        "param_dtype",
        "param_name",
        "occurrence_index_within_recording_mode",
        "param_count_declared",
        "value_class",
        "zero_nonzero_pattern",
        "parsed_values",
        "raw_values",
        "n_recordings",
        "n_unique_parsed_values",
        "stability_label",
        "example_parsed_values",
        "extraction_status",
        "header_token",
    ]

    preferred_cols = [col for col in preferred_cols if col in candidate_df.columns]
    remaining_cols = [col for col in candidate_df.columns if col not in preferred_cols]
    candidate_df = candidate_df[preferred_cols + remaining_cols]
    candidate_df = candidate_df.sort_values(
        [
            "candidate_family",
            "mode",
            "param_name",
            "occurrence_index_within_recording_mode",
            "recording_id",
        ]
    ).reset_index(drop=True)

    return candidate_df


def build_feparam_candidate_metadata_summary(candidate_long_df):
    """
    Build summary of candidate acquisition metadata.

    Parameters
    ----------
    candidate_long_df : pandas.DataFrame
        Long-form candidate metadata table.

    Returns
    -------
    pandas.DataFrame
        Candidate metadata summary table.
    """
    if len(candidate_long_df) == 0:
        return pd.DataFrame()

    rows = []

    group_cols = [
        "candidate_family",
        "mode",
        "param_dtype",
        "param_name",
        "occurrence_index_within_recording_mode",
    ]

    for group_key, group_df in candidate_long_df.groupby(group_cols):
        (
            candidate_family,
            mode,
            param_dtype,
            param_name,
            occurrence_index,
        ) = group_key

        recording_ids = sorted(group_df["recording_id"].unique().tolist())
        parsed_values_unique = sorted(
            group_df["parsed_values"].fillna("").astype(str).unique().tolist()
        )
        zero_patterns = sorted(
            group_df["zero_nonzero_pattern"].fillna("").astype(str).unique().tolist()
        )
        stability_labels = sorted(
            group_df["stability_label"].fillna("").astype(str).unique().tolist()
        )
        value_classes = sorted(
            group_df["value_class"].fillna("").astype(str).unique().tolist()
        )

        rows.append(
            {
                "candidate_family": candidate_family,
                "mode": mode,
                "param_dtype": param_dtype,
                "param_name": param_name,
                "occurrence_index_within_recording_mode": occurrence_index,
                "n_recordings": len(recording_ids),
                "n_unique_parsed_values": len(parsed_values_unique),
                "stability_labels": ";".join(stability_labels),
                "value_classes": ";".join(value_classes),
                "zero_nonzero_patterns": ";".join(zero_patterns),
                "recording_ids": ";".join(recording_ids),
                "example_parsed_values": " || ".join(parsed_values_unique[:8]),
            }
        )

    summary_df = pd.DataFrame(rows)

    summary_df = summary_df.sort_values(
        [
            "candidate_family",
            "mode",
            "n_recordings",
            "n_unique_parsed_values",
            "param_name",
        ],
        ascending=[True, True, False, False, True],
    ).reset_index(drop=True)

    return summary_df


def build_feparam_candidate_metadata_wide(candidate_long_df):
    """
    Build wide recording-level candidate metadata table.

    Parameters
    ----------
    candidate_long_df : pandas.DataFrame
        Long-form candidate metadata table.

    Returns
    -------
    pandas.DataFrame
        Wide table, one row per recording.
    """
    if len(candidate_long_df) == 0:
        return pd.DataFrame()

    value_df = candidate_long_df[
        [
            "recording_id",
            "candidate_key",
            "parsed_values",
        ]
    ].copy()

    # If duplicates remain, keep the first deterministic value.
    value_df = value_df.drop_duplicates(subset=["recording_id", "candidate_key"], keep="first",)
    wide_df = value_df.pivot(index="recording_id", columns="candidate_key", values="parsed_values",).reset_index()
    wide_df.columns.name = None

    return wide_df


def summarize_feparam_candidate_metadata(candidate_long_df, candidate_summary_df, candidate_wide_df):
    """
    Print summary of FeParam candidate metadata outputs.

    Parameters
    ----------
    candidate_long_df : pandas.DataFrame
        Candidate long table.

    candidate_summary_df : pandas.DataFrame
        Candidate summary table.

    candidate_wide_df : pandas.DataFrame
        Candidate wide table.
    """
    print("FeParam candidate acquisition metadata")
    print("=" * 80)
    print(f"Candidate long rows: {len(candidate_long_df)}")
    print(f"Candidate summary rows: {len(candidate_summary_df)}")
    print(f"Candidate wide shape: {candidate_wide_df.shape}")
    print()
    print("Candidate rows by family:")
    print(candidate_long_df["candidate_family"].value_counts())
    print()
    print("Candidate unique parameters by family/mode:")
    display(candidate_summary_df.groupby(["candidate_family", "mode"])["param_name"].nunique().reset_index(name="n_unique_param_names"))
    print()
    print("Variable 10/10 candidate examples:")
    variable_10_df = candidate_summary_df[candidate_summary_df["stability_labels"].str.contains("variable_10_of_10", na=False)].copy()
    display(variable_10_df.head(50))
    print()
    print("Stable 10/10 candidate examples:")
    stable_10_df = candidate_summary_df[candidate_summary_df["stability_labels"].str.contains("stable_10_of_10", na=False)].copy()
    display(stable_10_df.head(50))


def save_feparam_candidate_metadata_outputs(candidate_long_df, candidate_summary_df, candidate_wide_df, output_dir):
    """
    Save FeParam candidate metadata outputs.

    Parameters
    ----------
    candidate_long_df : pandas.DataFrame
        Candidate long table.

    candidate_summary_df : pandas.DataFrame
        Candidate summary table.

    candidate_wide_df : pandas.DataFrame
        Candidate wide table.

    output_dir : pathlib.Path
        Output directory.

    Returns
    -------
    dict
        Saved output paths.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    long_path = output_dir / "nb08_feparam_candidate_metadata_long.csv"
    summary_path = output_dir / "nb08_feparam_candidate_metadata_summary.csv"
    wide_path = output_dir / "nb08_feparam_candidate_metadata_wide.csv"
    candidate_long_df.to_csv(long_path, index=False)
    candidate_summary_df.to_csv(summary_path, index=False)
    candidate_wide_df.to_csv(wide_path, index=False)
    print(f"Saved candidate metadata long CSV: {long_path}")
    print(f"Saved candidate metadata summary CSV: {summary_path}")
    print(f"Saved candidate metadata wide CSV: {wide_path}")

    return {
        "candidate_metadata_long_csv": long_path,
        "candidate_metadata_summary_csv": summary_path,
        "candidate_metadata_wide_csv": wide_path,
    }

In [16]:
feparam_candidate_long_df = build_feparam_candidate_metadata_long(value_df=feparam_raw_value_df, value_summary_df=feparam_value_summary_df,)
feparam_candidate_summary_df = build_feparam_candidate_metadata_summary(feparam_candidate_long_df)
feparam_candidate_wide_df = build_feparam_candidate_metadata_wide(feparam_candidate_long_df)
summarize_feparam_candidate_metadata(
    candidate_long_df=feparam_candidate_long_df,
    candidate_summary_df=feparam_candidate_summary_df,
    candidate_wide_df=feparam_candidate_wide_df,
)

feparam_candidate_output_paths = save_feparam_candidate_metadata_outputs(
    candidate_long_df=feparam_candidate_long_df,
    candidate_summary_df=feparam_candidate_summary_df,
    candidate_wide_df=feparam_candidate_wide_df,
    output_dir=PATHS["NB08_REPORTS_DIR"],
)
feparam_candidate_summary_df

FeParam candidate acquisition metadata
Candidate long rows: 2587
Candidate summary rows: 330
Candidate wide shape: (10, 331)

Candidate rows by family:
candidate_family
angle_steering                     750
depth_range_geometry               567
mode_optimization_dynamic_range    340
threshold_flow_processing          330
frequency_prf                      190
frame_rate_persistence             180
gain_tgc_lgc                       160
focus                               50
power                               20
Name: count, dtype: int64

Candidate unique parameters by family/mode:


,candidate_family,mode,n_unique_param_names
0,angle_steering,BC,47
1,angle_steering,PW,6
2,depth_range_geometry,BC,57
3,depth_range_geometry,PW,19
4,focus,BC,5
5,focus,PW,2
6,frame_rate_persistence,BC,18
7,frame_rate_persistence,PW,3
8,frequency_prf,BC,19
9,frequency_prf,PW,15



Variable 10/10 candidate examples:


,candidate_family,mode,param_dtype,param_name,occurrence_index_within_recording_mode,n_recordings,n_unique_parsed_values,stability_labels,value_classes,zero_nonzero_patterns,recording_ids,example_parsed_values
81,depth_range_geometry,BC,double,CDispLineGap,1,10,2,variable_10_of_10,int_scalar;numeric_scalar,all_nonzero_or_true;all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,0 || 0.1598326383649555
82,depth_range_geometry,BC,uint16,CMaxDepthUploadPointNum,1,10,2,variable_10_of_10,int_scalar,all_nonzero_or_true;all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,0 || 620
83,depth_range_geometry,BC,double,CMaxLineCount,1,10,2,variable_10_of_10,int_scalar,all_nonzero_or_true;all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,0 || 240
84,depth_range_geometry,BC,int16,CUploadLineRange,1,10,2,variable_10_of_10,int_array,all_nonzero_or_true;all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,-60|59 || 0|0
85,depth_range_geometry,BC,uint16,CUploadPointRange,1,10,2,variable_10_of_10,int_array,all_nonzero_or_true;all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,0|0 || 152|392
164,frame_rate_persistence,BC,bool,CAdaptPersistenceOn,1,10,2,variable_10_of_10,bool_scalar,all_nonzero_or_true;all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,False || True
165,frame_rate_persistence,BC,uint8,CFrameCompCoef,1,10,2,variable_10_of_10,int_array,all_nonzero_or_true;all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,0|0|0 || 128|128|128
166,frame_rate_persistence,BC,bool,CFrameCompOn,1,10,2,variable_10_of_10,bool_scalar,all_nonzero_or_true;all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,False || True
167,frame_rate_persistence,BC,int32,CPersistLevel,1,10,2,variable_10_of_10,int_scalar,all_nonzero_or_true,202606130411060002SMP;202606130413540003SMP;20...,3 || 4
168,frame_rate_persistence,BC,bool,Cpowpersistent,1,10,2,variable_10_of_10,bool_scalar,all_nonzero_or_true;all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,False || True



Stable 10/10 candidate examples:


,candidate_family,mode,param_dtype,param_name,occurrence_index_within_recording_mode,n_recordings,n_unique_parsed_values,stability_labels,value_classes,zero_nonzero_patterns,recording_ids,example_parsed_values
0,angle_steering,BC,float,Angle,1,10,1,stable_10_of_10,numeric_scalar,all_nonzero_or_true,202606130411060002SMP;202606130413540003SMP;20...,-0.2094399929046631
1,angle_steering,BC,float,Angle,2,10,1,stable_10_of_10,int_scalar,all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,0
2,angle_steering,BC,float,Angle,3,10,1,stable_10_of_10,numeric_scalar,all_nonzero_or_true,202606130411060002SMP;202606130413540003SMP;20...,0.2094399929046631
3,angle_steering,BC,float,Angle,4,10,1,stable_10_of_10,int_scalar,all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,0
4,angle_steering,BC,float,Angle,5,10,1,stable_10_of_10,int_scalar,all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,0
5,angle_steering,BC,float,Angle,6,10,1,stable_10_of_10,int_scalar,all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,0
6,angle_steering,BC,float,Angle,7,10,1,stable_10_of_10,int_scalar,all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,0
7,angle_steering,BC,float,Angle,8,10,1,stable_10_of_10,int_scalar,all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,0
8,angle_steering,BC,float,Angle,9,10,1,stable_10_of_10,int_scalar,all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,0
9,angle_steering,BC,float,Angle,10,10,1,stable_10_of_10,int_scalar,all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,0


Saved candidate metadata long CSV: E:\DopplerLab\reports\nb08_feparam\nb08_feparam_candidate_metadata_long.csv
Saved candidate metadata summary CSV: E:\DopplerLab\reports\nb08_feparam\nb08_feparam_candidate_metadata_summary.csv
Saved candidate metadata wide CSV: E:\DopplerLab\reports\nb08_feparam\nb08_feparam_candidate_metadata_wide.csv


,candidate_family,mode,param_dtype,param_name,occurrence_index_within_recording_mode,n_recordings,n_unique_parsed_values,stability_labels,value_classes,zero_nonzero_patterns,recording_ids,example_parsed_values
0,angle_steering,BC,float,Angle,1,10,1,stable_10_of_10,numeric_scalar,all_nonzero_or_true,202606130411060002SMP;202606130413540003SMP;20...,-0.2094399929046631
1,angle_steering,BC,float,Angle,2,10,1,stable_10_of_10,int_scalar,all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,0
2,angle_steering,BC,float,Angle,3,10,1,stable_10_of_10,numeric_scalar,all_nonzero_or_true,202606130411060002SMP;202606130413540003SMP;20...,0.2094399929046631
3,angle_steering,BC,float,Angle,4,10,1,stable_10_of_10,int_scalar,all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,0
4,angle_steering,BC,float,Angle,5,10,1,stable_10_of_10,int_scalar,all_zero_or_false,202606130411060002SMP;202606130413540003SMP;20...,0
...,...,...,...,...,...,...,...,...,...,...,...,...
325,threshold_flow_processing,PW,uint8,PWFDS_CutBit,1,7,1,stable_subset,int_scalar,all_nonzero_or_true,202606130411060002SMP;202606130413540003SMP;20...,10
326,threshold_flow_processing,PW,float,PwWFTran,1,7,1,stable_subset,int_scalar,all_nonzero_or_true,202606130411060002SMP;202606130413540003SMP;20...,2
327,threshold_flow_processing,PW,float,PwWallFilterCoef,1,7,1,stable_subset,numeric_array,all_nonzero_or_true,202606130411060002SMP;202606130413540003SMP;20...,0.8109620213508606|-1.627869963645935|0.816917...
328,threshold_flow_processing,PW,float,L1_e_C_ThrePCT,1,3,1,stable_subset,int_scalar,all_zero_or_false,202606130417060004SMP;202606130422440006SMP;20...,0


## Final validation and summary

This section validates the NB08 FeParam extraction checkpoint and writes a compact summary report.

NB08 is frozen as an exploratory metadata extraction notebook.

The confirmed value of NB08 is:

- FeParam blobs can be extracted from `VirtualMachine.bin`,
- readable parameter-like strings are present,
- raw scalar/array values can be extracted experimentally,
- candidate acquisition metadata tables can be generated.

Important:

No clinical meaning is assigned to these parameters.
No Doppler velocity or hemodynamic feature is extracted from FeParam.

In [18]:
def validate_nb08_feparam_checkpoint(
    blob_inventory_df,
    string_inventory_df,
    parameter_summary_df,
    raw_value_df,
    value_summary_df,
    candidate_long_df,
    candidate_summary_df,
    candidate_wide_df,
):
    """
    Validate the NB08 FeParam extraction checkpoint.

    This validation checks whether the notebook produced the expected
    exploratory metadata outputs for the current native batch.

    Parameters
    ----------
    blob_inventory_df : pandas.DataFrame
        FeParam blob inventory.

    string_inventory_df : pandas.DataFrame
        Readable string inventory.

    parameter_summary_df : pandas.DataFrame
        Parameter-name summary.

    raw_value_df : pandas.DataFrame
        Raw value extraction table.

    value_summary_df : pandas.DataFrame
        Value stability summary.

    candidate_long_df : pandas.DataFrame
        Candidate metadata long table.

    candidate_summary_df : pandas.DataFrame
        Candidate metadata summary table.

    candidate_wide_df : pandas.DataFrame
        Candidate metadata wide table.

    Returns
    -------
    pandas.DataFrame
        Validation checks.
    """
    checks = []
    def add_check(check_name, passed, observed, expected, severity="error"):
        checks.append(
            {
                "check_name": check_name,
                "passed": bool(passed),
                "observed": observed,
                "expected": expected,
                "severity": severity,
            }
        )

    add_check(
        check_name="blob_inventory_rows",
        passed=len(blob_inventory_df) == 20,
        observed=len(blob_inventory_df),
        expected=20,
    )
    add_check(
        check_name="all_blobs_extract_ok",
        passed=bool(blob_inventory_df["blob_extract_ok"].all()),
        observed=int(blob_inventory_df["blob_extract_ok"].sum()),
        expected=20,
    )
    pw_offsets = sorted(
        blob_inventory_df[blob_inventory_df["mode"] == "PW"]["feparam_offset"]
        .dropna()
        .unique()
        .tolist()
    )

    add_check(
        check_name="pw_offset_zero",
        passed=pw_offsets == [0],
        observed=str(pw_offsets),
        expected="[0]",
    )
    bc_offsets = sorted(
        blob_inventory_df[blob_inventory_df["mode"] == "BC"]["feparam_offset"]
        .dropna()
        .unique()
        .tolist()
    )
    add_check(
        check_name="bc_offset_4530",
        passed=bc_offsets == [4530],
        observed=str(bc_offsets),
        expected="[4530]",
    )
    add_check(
        check_name="readable_strings_present",
        passed=len(string_inventory_df) > 1000,
        observed=len(string_inventory_df),
        expected=">1000",
    )
    parameter_like_count = int(string_inventory_df["is_parameter_like"].sum())

    add_check(
        check_name="parameter_like_strings_present",
        passed=parameter_like_count > 500,
        observed=parameter_like_count,
        expected=">500",
    )
    unique_pw_params = int(
        parameter_summary_df[parameter_summary_df["mode"] == "PW"]["param_name"]
        .nunique()
    )
    unique_bc_params = int(
        parameter_summary_df[parameter_summary_df["mode"] == "BC"]["param_name"]
        .nunique()
    )
    add_check(
        check_name="pw_unique_parameter_names_present",
        passed=unique_pw_params > 50,
        observed=unique_pw_params,
        expected=">50",
    )
    add_check(
        check_name="bc_unique_parameter_names_present",
        passed=unique_bc_params > 200,
        observed=unique_bc_params,
        expected=">200",
    )
    add_check(
        check_name="raw_value_rows_present",
        passed=len(raw_value_df) > 500,
        observed=len(raw_value_df),
        expected=">500",
    )
    stable_10_count = int(
        (value_summary_df["stability_label"] == "stable_10_of_10").sum()
    )
    variable_10_count = int(
        (value_summary_df["stability_label"] == "variable_10_of_10").sum()
    )
    add_check(
        check_name="stable_10_of_10_values_present",
        passed=stable_10_count > 100,
        observed=stable_10_count,
        expected=">100",
    )
    add_check(
        check_name="variable_10_of_10_values_present",
        passed=variable_10_count > 0,
        observed=variable_10_count,
        expected=">0",
    )
    add_check(
        check_name="candidate_metadata_long_rows_present",
        passed=len(candidate_long_df) > 1000,
        observed=len(candidate_long_df),
        expected=">1000",
    )
    add_check(
        check_name="candidate_metadata_summary_rows_present",
        passed=len(candidate_summary_df) > 100,
        observed=len(candidate_summary_df),
        expected=">100",
    )
    add_check(
        check_name="candidate_wide_has_10_recordings",
        passed=candidate_wide_df.shape[0] == 10,
        observed=candidate_wide_df.shape[0],
        expected=10,
    )
    required_families = {
        "power",
        "gain_tgc_lgc",
        "focus",
        "frequency_prf",
        "angle_steering",
        "depth_range_geometry",
        "frame_rate_persistence",
        "mode_optimization_dynamic_range",
        "threshold_flow_processing",
    }
    observed_families = set(candidate_long_df["candidate_family"].unique().tolist())
    missing_families = sorted(required_families - observed_families)
    add_check(
        check_name="all_expected_candidate_families_present",
        passed=len(missing_families) == 0,
        observed=";".join(sorted(observed_families)),
        expected=";".join(sorted(required_families)),
    )

    return pd.DataFrame(checks)


def get_top_variable_candidate_examples(candidate_summary_df, max_rows=20):
    """
    Get compact top variable candidate examples.

    Parameters
    ----------
    candidate_summary_df : pandas.DataFrame
        Candidate metadata summary table.

    max_rows : int
        Maximum rows.

    Returns
    -------
    pandas.DataFrame
        Top variable candidate examples.
    """
    variable_df = candidate_summary_df[candidate_summary_df["stability_labels"].str.contains("variable_10_of_10",na=False,)].copy()

    useful_family_order = {
        "gain_tgc_lgc": 0,
        "frequency_prf": 1,
        "frame_rate_persistence": 2,
        "depth_range_geometry": 3,
        "mode_optimization_dynamic_range": 4,
        "threshold_flow_processing": 5,
        "focus": 6,
        "power": 7,
        "angle_steering": 8,
    }

    variable_df["family_order"] = variable_df["candidate_family"].map(useful_family_order).fillna(99)

    variable_df = variable_df.sort_values(
        [
            "family_order",
            "candidate_family",
            "mode",
            "param_name",
            "occurrence_index_within_recording_mode",
        ]
    ).reset_index(drop=True)

    keep_cols = [
        "candidate_family",
        "mode",
        "param_dtype",
        "param_name",
        "occurrence_index_within_recording_mode",
        "n_recordings",
        "n_unique_parsed_values",
        "example_parsed_values",
    ]

    keep_cols = [col for col in keep_cols if col in variable_df.columns]
    return variable_df[keep_cols].head(max_rows)


def write_nb08_feparam_summary_markdown(
    blob_inventory_df,
    string_inventory_df,
    parameter_summary_df,
    raw_value_df,
    value_summary_df,
    candidate_long_df,
    candidate_summary_df,
    candidate_wide_df,
    validation_df,
    output_path,
):
    """
    Write NB08 FeParam summary markdown.

    Parameters
    ----------
    blob_inventory_df : pandas.DataFrame
        Blob inventory.

    string_inventory_df : pandas.DataFrame
        String inventory.

    parameter_summary_df : pandas.DataFrame
        Parameter summary.

    raw_value_df : pandas.DataFrame
        Raw value table.

    value_summary_df : pandas.DataFrame
        Value stability summary.

    candidate_long_df : pandas.DataFrame
        Candidate metadata long table.

    candidate_summary_df : pandas.DataFrame
        Candidate metadata summary table.

    candidate_wide_df : pandas.DataFrame
        Candidate metadata wide table.

    validation_df : pandas.DataFrame
        Validation checks.

    output_path : pathlib.Path or str
        Markdown output path.

    Returns
    -------
    pathlib.Path
        Saved markdown path.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    validation_pass = bool(validation_df["passed"].all())
    blob_ok_count = int(blob_inventory_df["blob_extract_ok"].sum())
    parameter_like_count = int(string_inventory_df["is_parameter_like"].sum())
    unique_pw_params = int(parameter_summary_df[parameter_summary_df["mode"] == "PW"]["param_name"].nunique())
    unique_bc_params = int(parameter_summary_df[parameter_summary_df["mode"] == "BC"]["param_name"].nunique())
    stability_counts = (value_summary_df["stability_label"].value_counts().to_dict())
    candidate_family_counts = (candidate_long_df["candidate_family"].value_counts().to_dict())
    variable_examples_df = get_top_variable_candidate_examples(candidate_summary_df, max_rows=20,)

    lines = []

    lines.append("# NB08 FeParam Metadata Extraction Summary")
    lines.append("")
    lines.append("## Status")
    lines.append("")
    lines.append(f"- Validation pass: `{validation_pass}`")
    lines.append(f"- FeParam blob rows: `{len(blob_inventory_df)}`")
    lines.append(f"- Blob extraction OK: `{blob_ok_count}/{len(blob_inventory_df)}`")
    lines.append("- Layout status: `PW offset 0`, `BC offset 4530`, BC size derived from `VirtualMachine.bin` size")
    lines.append("- Layout caveat: `VirtualMachine.txt` param-size parsing was not fully decoded; current layout uses Scope04/file-consistent fallback.")
    lines.append("")
    lines.append("## String and parameter inventory")
    lines.append("")
    lines.append(f"- Readable strings: `{len(string_inventory_df)}`")
    lines.append(f"- Parameter-like strings: `{parameter_like_count}`")
    lines.append(f"- Unique PW parameter names: `{unique_pw_params}`")
    lines.append(f"- Unique BC parameter names: `{unique_bc_params}`")
    lines.append("")
    lines.append("## Raw value extraction")
    lines.append("")
    lines.append(f"- Parameter occurrences extracted: `{len(raw_value_df)}`")
    lines.append("")
    lines.append("Value stability counts:")
    lines.append("")

    for label, count in stability_counts.items():
        lines.append(f"- `{label}`: `{count}`")

    lines.append("")
    lines.append("## Candidate acquisition metadata")
    lines.append("")
    lines.append(f"- Candidate long rows: `{len(candidate_long_df)}`")
    lines.append(f"- Candidate summary rows: `{len(candidate_summary_df)}`")
    lines.append(f"- Candidate wide shape: `{candidate_wide_df.shape[0]} x {candidate_wide_df.shape[1]}`")
    lines.append("")
    lines.append("Candidate rows by family:")
    lines.append("")

    for family, count in candidate_family_counts.items():
        lines.append(f"- `{family}`: `{count}`")

    lines.append("")
    lines.append("## Example variable 10/10 metadata candidates")
    lines.append("")

    if len(variable_examples_df) == 0:
        lines.append("- None found.")
    else:
        for _, row in variable_examples_df.iterrows():
            lines.append(
                f"- `{row['candidate_family']}` / `{row['mode']}` / "
                f"`{row['param_name']}` occurrence `{row['occurrence_index_within_recording_mode']}`: "
                f"`{row['example_parsed_values']}`"
            )

    lines.append("")
    lines.append("## Interpretation boundary")
    lines.append("")
    lines.append("FeParam appears to contain a readable parameter tree and can be useful for acquisition metadata comparison.")
    lines.append("")
    lines.append("However, NB08 does **not** assign clinical meaning to these parameters.")
    lines.append("")
    lines.append("NB08 does **not** extract:")
    lines.append("")
    lines.append("- Doppler velocity envelope")
    lines.append("- PSV / EDV / RI / PI / VTI")
    lines.append("- blood pressure")
    lines.append("- stroke volume")
    lines.append("- cardiac output")
    lines.append("- arterial stiffness or compliance")
    lines.append("- native beat timing")
    lines.append("")
    lines.append("## Frozen checkpoint")
    lines.append("")
    lines.append("NB08 is frozen as an exploratory FeParam metadata extraction checkpoint.")
    lines.append("")
    lines.append("Future work may manually validate selected parameters against machine settings or controlled acquisitions.")

    output_path.write_text("\n".join(lines), encoding="utf-8",)

    return output_path


def save_nb08_final_outputs(
    validation_df,
    blob_inventory_df,
    string_inventory_df,
    parameter_summary_df,
    raw_value_df,
    value_summary_df,
    candidate_long_df,
    candidate_summary_df,
    candidate_wide_df,
    paths,
):
    """
    Save NB08 final validation and summary outputs.

    Parameters
    ----------
    validation_df : pandas.DataFrame
        Validation checks.

    blob_inventory_df : pandas.DataFrame
        Blob inventory.

    string_inventory_df : pandas.DataFrame
        String inventory.

    parameter_summary_df : pandas.DataFrame
        Parameter summary.

    raw_value_df : pandas.DataFrame
        Raw value table.

    value_summary_df : pandas.DataFrame
        Value stability summary.

    candidate_long_df : pandas.DataFrame
        Candidate metadata long table.

    candidate_summary_df : pandas.DataFrame
        Candidate metadata summary table.

    candidate_wide_df : pandas.DataFrame
        Candidate metadata wide table.

    paths : dict
        Path dictionary.

    Returns
    -------
    dict
        Saved paths.
    """
    reports_dir = Path(paths["NB08_REPORTS_DIR"])
    docs_dir = Path(paths["NB08_DOCS_DIR"])
    reports_dir.mkdir(parents=True, exist_ok=True)
    docs_dir.mkdir(parents=True, exist_ok=True)
    validation_path = reports_dir / "nb08_feparam_validation_checks.csv"
    summary_path = docs_dir / "nb08_feparam_summary.md"
    validation_df.to_csv(validation_path, index=False)

    write_nb08_feparam_summary_markdown(
        blob_inventory_df=blob_inventory_df,
        string_inventory_df=string_inventory_df,
        parameter_summary_df=parameter_summary_df,
        raw_value_df=raw_value_df,
        value_summary_df=value_summary_df,
        candidate_long_df=candidate_long_df,
        candidate_summary_df=candidate_summary_df,
        candidate_wide_df=candidate_wide_df,
        validation_df=validation_df,
        output_path=summary_path,
    )

    print(f"Saved NB08 validation checks CSV: {validation_path}")
    print(f"Saved NB08 summary markdown: {summary_path}")

    return {
        "validation_csv": validation_path,
        "summary_markdown": summary_path,
    }


In [19]:
nb08_validation_df = validate_nb08_feparam_checkpoint(
    blob_inventory_df=feparam_blob_inventory_df,
    string_inventory_df=feparam_string_inventory_df,
    parameter_summary_df=feparam_parameter_summary_df,
    raw_value_df=feparam_raw_value_df,
    value_summary_df=feparam_value_summary_df,
    candidate_long_df=feparam_candidate_long_df,
    candidate_summary_df=feparam_candidate_summary_df,
    candidate_wide_df=feparam_candidate_wide_df,
)
print("NB08 validation")
print("=" * 80)

if nb08_validation_df["passed"].all():
    print("PASS — all NB08 validation checks passed.")
else:
    print("WARNING — at least one NB08 validation check failed.")
    display(nb08_validation_df[~nb08_validation_df["passed"]])

print()
display(nb08_validation_df)

top_variable_candidate_examples_df = get_top_variable_candidate_examples(feparam_candidate_summary_df, max_rows=20,)

print()
print("Top variable candidate examples")
print("=" * 80)
display(top_variable_candidate_examples_df)

nb08_final_output_paths = save_nb08_final_outputs(
    validation_df=nb08_validation_df,
    blob_inventory_df=feparam_blob_inventory_df,
    string_inventory_df=feparam_string_inventory_df,
    parameter_summary_df=feparam_parameter_summary_df,
    raw_value_df=feparam_raw_value_df,
    value_summary_df=feparam_value_summary_df,
    candidate_long_df=feparam_candidate_long_df,
    candidate_summary_df=feparam_candidate_summary_df,
    candidate_wide_df=feparam_candidate_wide_df,
    paths=PATHS,
)

NB08 validation
PASS — all NB08 validation checks passed.



,check_name,passed,observed,expected,severity
0,blob_inventory_rows,True,20,20,error
1,all_blobs_extract_ok,True,20,20,error
2,pw_offset_zero,True,[0],[0],error
3,bc_offset_4530,True,[4530],[4530],error
4,readable_strings_present,True,8467,>1000,error
5,parameter_like_strings_present,True,6787,>500,error
6,pw_unique_parameter_names_present,True,143,>50,error
7,bc_unique_parameter_names_present,True,584,>200,error
8,raw_value_rows_present,True,6787,>500,error
9,stable_10_of_10_values_present,True,496,>100,error



Top variable candidate examples


,candidate_family,mode,param_dtype,param_name,occurrence_index_within_recording_mode,n_recordings,n_unique_parsed_values,example_parsed_values
0,gain_tgc_lgc,BC,int32,CGainLevel,1,10,2,25 || 29
1,frequency_prf,BC,int8,CDispFreq,1,10,2,0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0 || 53|...
2,frequency_prf,BC,double,CPrf,1,10,2,0 || 1082.2510822510822
3,frequency_prf,BC,float,CTxFreq,1,10,2,0 || 5715000
4,frequency_prf,BC,double,CTxPrf,1,10,2,0 || 11904.761904761905
5,frame_rate_persistence,BC,bool,CAdaptPersistenceOn,1,10,2,False || True
6,frame_rate_persistence,BC,uint8,CFrameCompCoef,1,10,2,0|0|0 || 128|128|128
7,frame_rate_persistence,BC,bool,CFrameCompOn,1,10,2,False || True
8,frame_rate_persistence,BC,int32,CPersistLevel,1,10,2,3 || 4
9,frame_rate_persistence,BC,bool,Cpowpersistent,1,10,2,False || True


Saved NB08 validation checks CSV: E:\DopplerLab\reports\nb08_feparam\nb08_feparam_validation_checks.csv
Saved NB08 summary markdown: E:\DopplerLab\docs\nb08_feparam\nb08_feparam_summary.md
